In [14]:
import tensorflow as tf
from tensorflow.keras.layers import Input,Multiply,Activation,Reshape,Conv2D,MaxPooling2D,DepthwiseConv2D,Concatenate,GlobalAveragePooling2D,Dense,BatchNormalization,Add,Resizing, Rescaling,ReLU,AveragePooling2D,Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import RandomFlip, RandomRotation, RandomTranslation, RandomZoom


In [15]:
import tensorflow as tf

train_path = "/kaggle/input/datasets/ayush1220/cifar10/cifar10/train"
test_path = "/kaggle/input/datasets/ayush1220/cifar10/cifar10/test"

train_ds = tf.keras.utils.image_dataset_from_directory(
    train_path,
    image_size=(32, 32),      
    batch_size=64,
    label_mode="int",
    shuffle=True
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    test_path,
    image_size=(32, 32),
    batch_size=64,
    label_mode="int",
    shuffle=False
)

Found 50000 files belonging to 10 classes.
Found 10000 files belonging to 10 classes.


In [16]:
train_ds = train_ds.map(lambda x, y: (tf.cast(x, tf.float32) / 255.0, y))
test_ds = test_ds.map(lambda x, y: (tf.cast(x, tf.float32) / 255.0, y))

### Inception- Model

In [17]:
def inception_block(x, f1, f3, f5, fp):

    b1 = Conv2D(f1, (1,1), padding="same")(x)
    b1 = BatchNormalization(momentum=0.9)(b1)
    b1 = ReLU()(b1)

    
    b2 = Conv2D(f3//2, (1,1), padding="same")(x)
    b2 = BatchNormalization(momentum=0.9)(b2)
    b2 = ReLU()(b2)

    b2 = Conv2D(f3, (3,3), padding="same")(b2)
    b2 = BatchNormalization(momentum=0.9)(b2)
    b2 = ReLU()(b2)

  
    b3 = Conv2D(f5//2, (1,1), padding="same")(x)
    b3 = BatchNormalization(momentum=0.9)(b3)
    b3 = ReLU()(b3)
    b3 = Conv2D(f5, (5,5), padding="same")(b3)
    b3 = BatchNormalization(momentum=0.9)(b3)
    b3 = ReLU()(b3)

    
    b4 = AveragePooling2D((3,3), strides=1, padding="same")(x)
    b4 = Conv2D(fp, (1,1), padding="same")(b4)
    b4 = BatchNormalization(momentum=0.9)(b4)
    b4 = ReLU()(b4)

    return Concatenate()([b1, b2, b3, b4])

In [18]:
inputs = Input(shape=(32,32,3))

x = RandomFlip("horizontal")(inputs)
x = RandomRotation(0.1)(x)
x = RandomTranslation(0.1, 0.1)(x)
x = RandomZoom(0.1)(x)

x = Conv2D(32, (3,3), padding="same")(x)
x = BatchNormalization()(x)
x = ReLU()(x)

x = Conv2D(64, (3,3), padding="same")(x)
x = BatchNormalization()(x)
x = ReLU()(x)

x = MaxPooling2D((2,2))(x)

x = inception_block(x,f1=32,f3=64,f5=32,fp=32)

x = inception_block(x,f1=64,f3=96,f5=48,fp=32)

x = MaxPooling2D((2,2))(x)

x = inception_block(x,f1=96,f3=128,f5=64,fp=64)

x = inception_block(x,f1=128,f3=192,f5=96,fp=64)

x = GlobalAveragePooling2D()(x)

x = Dropout(0.4)(x)

outputs = Dense(10, activation="softmax")(x)

model1 = Model(inputs, outputs)

model1.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_2       │ (None, 32, 32, 3) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ random_flip_1       │ (None, 32, 32, 3) │          0 │ input_layer_2[0]… │
│ (RandomFlip)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ random_rotation_1   │ (None, 32, 32, 3) │          0 │ random_flip_1[0]… │
│ (RandomRotation)    │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ random_translation… │ (None, 32, 32, 3) │          0 │ random_rotation_… │
│ (RandomTranslation) │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ random_zoom_1       │ (None, 32, 32, 3) │          0 │ random_translati… │
│ (RandomZoom)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_16 (Conv2D)  │ (None, 32, 32,    │        896 │ random_zoom_1[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 32, 32,    │        128 │ conv2d_16[0][0]   │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_14 (ReLU)     │ (None, 32, 32,    │          0 │ batch_normalizat… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_17 (Conv2D)  │ (None, 32, 32,    │     18,496 │ re_lu_14[0][0]    │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 32, 32,    │        256 │ conv2d_17[0][0]   │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_15 (ReLU)     │ (None, 32, 32,    │          0 │ batch_normalizat… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_3     │ (None, 16, 16,    │          0 │ re_lu_15[0][0]    │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_19 (Conv2D)  │ (None, 16, 16,    │      2,080 │ max_pooling2d_3[… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_21 (Conv2D)  │ (None, 16, 16,    │      1,040 │ max_pooling2d_3[… │
│                     │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 16, 16,    │        128 │ conv2d_19[0][0]   │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 16, 16,    │         64 │ conv2d_21[0][0]   │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_17 (ReLU)     │ (None, 16, 16,    │          0 │ batch_normalizat

 Total params: 753,826 (2.88 MB)

 Trainable params: 750,450 (2.86 MB)

 Non-trainable params: 3,376 (13.19 KB)

In [19]:
lr_schedule = tf.keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=1e-3, decay_steps=30*782
)

model1.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=lr_schedule),
    loss="sparse_categorical_crossentropy",metrics=["accuracy"]
)

In [20]:
model1.fit(train_ds,epochs=30)

Epoch 1/30
782/782 ━━━━━━━━━━━━━━━━━━━━ 57s 51ms/step - accuracy: 0.4531 - loss: 1.5108
Epoch 2/30
782/782 ━━━━━━━━━━━━━━━━━━━━ 40s 51ms/step - accuracy: 0.5880 - loss: 1.1561
Epoch 3/30
782/782 ━━━━━━━━━━━━━━━━━━━━ 39s 50ms/step - accuracy: 0.6506 - loss: 0.9948
Epoch 4/30
782/782 ━━━━━━━━━━━━━━━━━━━━ 39s 50ms/step - accuracy: 0.6913 - loss: 0.8816
Epoch 5/30
782/782 ━━━━━━━━━━━━━━━━━━━━ 40s 51ms/step - accuracy: 0.7181 - loss: 0.8076
Epoch 6/30
782/782 ━━━━━━━━━━━━━━━━━━━━ 39s 50ms/step - accuracy: 0.7435 - loss: 0.7427
Epoch 7/30
782/782 ━━━━━━━━━━━━━━━━━━━━ 39s 50ms/step - accuracy: 0.7556 - loss: 0.7039
Epoch 8/30
782/782 ━━━━━━━━━━━━━━━━━━━━ 39s 50ms/step - accuracy: 0.7733 - loss: 0.6536
Epoch 9/30
782/782 ━━━━━━━━━━━━━━━━━━━━ 39s 50ms/step - accuracy: 0.7842 - loss: 0.6245
Epoch 10/30
782/782 ━━━━━━━━━━━━━━━━━━━━ 39s 50ms/step - accuracy: 0.7958 - loss: 0.5903
Epoch 11/30
782/782 ━━━━━━━━━━━━━━━━━━━━ 39s 50ms/step - accuracy: 0.8060 - loss: 0.5585
Epoch 12/30
782/782 ━━━━━━━━━━

In [21]:
model1.evaluate(test_ds)

157/157 ━━━━━━━━━━━━━━━━━━━━ 7s 35ms/step - accuracy: 0.8611 - loss: 0.4276


[0.42764565348625183, 0.8611000180244446]

### ResNet

In [22]:
from tensorflow.keras.regularizers import l2

def basic_block(x, filters):
    shortcut = x

    y = Conv2D(filters, (3,3), padding="same", use_bias=False,
               kernel_regularizer=l2(1e-4))(x)
    y = BatchNormalization()(y)
    y = ReLU()(y)

    y = Conv2D(filters, (3,3), padding="same", use_bias=False,
               kernel_regularizer=l2(1e-4))(y)
    y = BatchNormalization()(y)

    if x.shape[-1] != filters:
        shortcut = Conv2D(filters, (1,1), padding="same", use_bias=False,
                           kernel_regularizer=l2(1e-5))(shortcut)
        shortcut = BatchNormalization()(shortcut)

    y = Add()([y, shortcut])
    y = ReLU()(y)

    return y

In [23]:
inputs = Input(shape=(32,32,3))

x = RandomFlip("horizontal")(inputs)
x = RandomRotation(0.1)(x)
x = RandomTranslation(0.1, 0.1)(x)
x = RandomZoom(0.1)(x)

x = Conv2D(32, (3,3), padding="same", use_bias=False)(x)
x = BatchNormalization()(x)
x = ReLU()(x)

x = Conv2D(64, (3,3), padding="same", use_bias=False)(x)
x = BatchNormalization()(x)
x = ReLU()(x)

x = MaxPooling2D((2,2))(x)

x = basic_block(x,64)
x = basic_block(x,64)

x = MaxPooling2D((2,2))(x)

x = basic_block(x,128)
x = basic_block(x,128)

x = MaxPooling2D((2,2))(x)

x = basic_block(x,256)
x = basic_block(x,256)

x = GlobalAveragePooling2D()(x)

x = Dropout(0.3)(x)

outputs = Dense(10, activation="softmax")(x)

model2 = Model(inputs, outputs)

model2.summary()

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_3       │ (None, 32, 32, 3) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ random_flip_2       │ (None, 32, 32, 3) │          0 │ input_layer_3[0]… │
│ (RandomFlip)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ random_rotation_2   │ (None, 32, 32, 3) │          0 │ random_flip_2[0]… │
│ (RandomRotation)    │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ random_translation… │ (None, 32, 32, 3) │          0 │ random_rotation_… │
│ (RandomTranslation) │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ random_zoom_2       │ (None, 32, 32, 3) │          0 │ random_translati… │
│ (RandomZoom)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_42 (Conv2D)  │ (None, 32, 32,    │        864 │ random_zoom_2[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 32, 32,    │        128 │ conv2d_42[0][0]   │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_40 (ReLU)     │ (None, 32, 32,    │          0 │ batch_normalizat… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_43 (Conv2D)  │ (None, 32, 32,    │     18,432 │ re_lu_40[0][0]    │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 32, 32,    │        256 │ conv2d_43[0][0]   │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_41 (ReLU)     │ (None, 32, 32,    │          0 │ batch_normalizat… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_5     │ (None, 16, 16,    │          0 │ re_lu_41[0][0]    │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_44 (Conv2D)  │ (None, 16, 16,    │     36,864 │ max_pooling2d_5[… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 16, 16,    │        256 │ conv2d_44[0][0]   │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_42 (ReLU)     │ (None, 16, 16,    │          0 │ batch_normalizat… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_45 (Conv2D)  │ (None, 16, 16,    │     36,864 │ re_lu_42[0][0]    │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 16, 16,    │        256 │ conv2d_45[0][0] 

 Total params: 2,799,850 (10.68 MB)

 Trainable params: 2,795,306 (10.66 MB)

 Non-trainable params: 4,544 (17.75 KB)

In [24]:
lr_schedule = tf.keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=1e-3, decay_steps=30*782)

model2.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=lr_schedule),
    loss="sparse_categorical_crossentropy",metrics=["accuracy"])

In [25]:
model2.fit(train_ds,epochs=30)

Epoch 1/30
782/782 ━━━━━━━━━━━━━━━━━━━━ 38s 37ms/step - accuracy: 0.4473 - loss: 1.7251
Epoch 2/30
782/782 ━━━━━━━━━━━━━━━━━━━━ 29s 37ms/step - accuracy: 0.5939 - loss: 1.3144
Epoch 3/30
782/782 ━━━━━━━━━━━━━━━━━━━━ 29s 36ms/step - accuracy: 0.6560 - loss: 1.1574
Epoch 4/30
782/782 ━━━━━━━━━━━━━━━━━━━━ 29s 36ms/step - accuracy: 0.6912 - loss: 1.0702
Epoch 5/30
782/782 ━━━━━━━━━━━━━━━━━━━━ 28s 36ms/step - accuracy: 0.7155 - loss: 1.0126
Epoch 6/30
782/782 ━━━━━━━━━━━━━━━━━━━━ 29s 37ms/step - accuracy: 0.7355 - loss: 0.9615
Epoch 7/30
782/782 ━━━━━━━━━━━━━━━━━━━━ 28s 36ms/step - accuracy: 0.7477 - loss: 0.9224
Epoch 8/30
782/782 ━━━━━━━━━━━━━━━━━━━━ 29s 37ms/step - accuracy: 0.7650 - loss: 0.8828
Epoch 9/30
782/782 ━━━━━━━━━━━━━━━━━━━━ 29s 37ms/step - accuracy: 0.7746 - loss: 0.8510
Epoch 10/30
782/782 ━━━━━━━━━━━━━━━━━━━━ 29s 37ms/step - accuracy: 0.7898 - loss: 0.8143
Epoch 11/30
782/782 ━━━━━━━━━━━━━━━━━━━━ 29s 37ms/step - accuracy: 0.7999 - loss: 0.7798
Epoch 12/30
782/782 ━━━━━━━━━━

In [26]:
model2.evaluate(test_ds)

157/157 ━━━━━━━━━━━━━━━━━━━━ 4s 24ms/step - accuracy: 0.8831 - loss: 0.5095


[0.5095087289810181, 0.8830999732017517]

### MobileNet

In [27]:
def mobilenet_block(x, filters, strides=1, dropout_rate=0):

    # Depthwise Convolution
    x = DepthwiseConv2D(kernel_size=3,strides=strides,padding="same",use_bias=False)(x)
    x = BatchNormalization()(x)
    x = ReLU(max_value=6)(x)

    if dropout_rate > 0:
        x = Dropout(dropout_rate)(x)

    # Pointwise Convolution
    x = Conv2D(filters, (1,1), padding="same", use_bias=False)(x)
    x = BatchNormalization()(x)
    x = ReLU(max_value=6)(x)

    return x

In [28]:
inputs = Input(shape=(32,32,3))

x = RandomFlip("horizontal")(inputs)
x = RandomRotation(0.1)(x)
x = RandomTranslation(0.1, 0.1)(x)
x = RandomZoom(0.1)(x)

x = Conv2D(32, (3,3), strides=2, padding="same", use_bias=False)(x)
x = BatchNormalization()(x)
x = ReLU(max_value=6)(x)

# MobileNet Blocks
x = mobilenet_block(x, 32)
x = mobilenet_block(x, 64)

x = mobilenet_block(x, 96, strides=2)

x = mobilenet_block(x, 96)

x = mobilenet_block(x, 160, strides=2, dropout_rate=0.1)

x = mobilenet_block(x, 160)

x = mobilenet_block(x, 256)

# Classifier
x = GlobalAveragePooling2D()(x)

x = Dropout(0.3)(x)

outputs = Dense(10, activation="softmax")(x)

model3 = Model(inputs, outputs)

model3.summary()

Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_4 (InputLayer)      │ (None, 32, 32, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_flip_3 (RandomFlip)      │ (None, 32, 32, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_rotation_3               │ (None, 32, 32, 3)      │             0 │
│ (RandomRotation)                │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_translation_3            │ (None, 32, 32, 3)      │             0 │
│ (RandomTranslation)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_zoom_3 (RandomZoom)      │ (None, 32, 32, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_58 (Conv2D)              │ (None, 16, 16, 32)     │           864 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_58          │ (None, 16, 16, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_54 (ReLU)                 │ (None, 16, 16, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ depthwise_conv2d                │ (None, 16, 16, 32)     │           288 │
│ (DepthwiseConv2D)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_59          │ (None, 16, 16, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_55 (ReLU)                 │ (None, 16, 16, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_59 (Conv2D)              │ (None, 16, 16, 32)     │         1,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_60          │ (None, 16, 16, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_56 (ReLU)                 │ (None, 16, 16, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ depthwise_conv2d_1              │ (None, 16, 16, 32)     │           288 │
│ (DepthwiseConv2D)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_61          │ (None, 16, 16, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_57 (ReLU)                 │ (None, 16, 16, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_60 (Conv2D)              │ (None, 16, 16, 64)     │         2,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_62          │ (None, 16, 16, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_58 (ReLU)                 │ (None, 16, 16, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼─────────────

 Total params: 115,690 (451.91 KB)

 Trainable params: 112,618 (439.91 KB)

 Non-trainable params: 3,072 (12.00 KB)

In [29]:
lr_schedule = tf.keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=1e-3, decay_steps=30*782
)

model3.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=lr_schedule),
    loss="sparse_categorical_crossentropy",metrics=["accuracy"]
)

In [30]:
model3.fit(train_ds,epochs=30)

Epoch 1/30


E0000 00:00:1785352986.484132      58 meta_optimizer.cc:967] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape inStatefulPartitionedCall/functional_3_1/dropout_3_1/stateless_dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer


782/782 ━━━━━━━━━━━━━━━━━━━━ 29s 26ms/step - accuracy: 0.3455 - loss: 1.7630
Epoch 2/30
782/782 ━━━━━━━━━━━━━━━━━━━━ 20s 25ms/step - accuracy: 0.4557 - loss: 1.4936
Epoch 3/30
782/782 ━━━━━━━━━━━━━━━━━━━━ 20s 25ms/step - accuracy: 0.5066 - loss: 1.3783
Epoch 4/30
782/782 ━━━━━━━━━━━━━━━━━━━━ 20s 25ms/step - accuracy: 0.5289 - loss: 1.3098
Epoch 5/30
782/782 ━━━━━━━━━━━━━━━━━━━━ 20s 26ms/step - accuracy: 0.5523 - loss: 1.2437
Epoch 6/30
782/782 ━━━━━━━━━━━━━━━━━━━━ 20s 25ms/step - accuracy: 0.5743 - loss: 1.1929
Epoch 7/30
782/782 ━━━━━━━━━━━━━━━━━━━━ 20s 25ms/step - accuracy: 0.5902 - loss: 1.1481
Epoch 8/30
782/782 ━━━━━━━━━━━━━━━━━━━━ 20s 25ms/step - accuracy: 0.6070 - loss: 1.1084
Epoch 9/30
782/782 ━━━━━━━━━━━━━━━━━━━━ 20s 25ms/step - accuracy: 0.6180 - loss: 1.0779
Epoch 10/30
782/782 ━━━━━━━━━━━━━━━━━━━━ 20s 25ms/step - accuracy: 0.6323 - loss: 1.0414
Epoch 11/30
782/782 ━━━━━━━━━━━━━━━━━━━━ 20s 25ms/step - accuracy: 0.6427 - loss: 1.0098
Epoch 12/30
782/782 ━━━━━━━━━━━━━━━━━━━━ 

In [31]:
model3.evaluate(test_ds)

157/157 ━━━━━━━━━━━━━━━━━━━━ 4s 23ms/step - accuracy: 0.7071 - loss: 0.8567


[0.856696367263794, 0.707099974155426]

### EfficientNet

In [32]:
def SE_block(inputs, ratio=4):

    filters = inputs.shape[-1]

    x = GlobalAveragePooling2D()(inputs)
    x = Reshape((1,1,filters))(x)

    x = Dense(max(1, filters//ratio), activation="swish")(x)
    x = Dense(filters, activation="sigmoid")(x)

    return Multiply()([inputs, x])


In [33]:
def MBConv_block(inputs,in_filters,out_filters,expand_ratio=4,strides=1):

    x = inputs
    expanded_filters = in_filters * expand_ratio

    if expand_ratio != 1:
        x = Conv2D(expanded_filters,1,padding="same",use_bias=False)(x)
        x = BatchNormalization()(x)
        x = Activation("swish")(x)

    x = DepthwiseConv2D(3,strides=strides,padding="same",use_bias=False)(x)
    x = BatchNormalization()(x)
    x = Activation("swish")(x)

    x = SE_block(x)

    x = Conv2D(out_filters,1,padding="same",use_bias=False)(x)
    x = BatchNormalization()(x)

    if strides == 1 and in_filters == out_filters:
        x = Add()([inputs, x])

    return x


In [34]:
inputs = Input(shape=(32,32,3))

x = RandomFlip("horizontal")(inputs)
x = RandomRotation(0.1)(x)
x = RandomZoom(0.1)(x)


x = Conv2D(32,3,padding="same",use_bias=False)(x)

x = BatchNormalization()(x)
x = Activation("swish")(x)

x = MBConv_block(x,in_filters=32,out_filters=16,expand_ratio=1)

x = MBConv_block(x,16,24,expand_ratio=4)

x = MBConv_block(x,24,24,expand_ratio=4)

x = MBConv_block(x,24,40,expand_ratio=4,strides=2)

x = MBConv_block(x,40,40,expand_ratio=4)

x = MBConv_block(x,40,64,expand_ratio=6,strides=2)

x = MBConv_block(x,64,64,expand_ratio=6)

x = Conv2D(128,1,padding="same",use_bias=False)(x)

x = BatchNormalization()(x)
x = Activation("swish")(x)

x = GlobalAveragePooling2D()(x)

x = Dropout(0.3)(x)

outputs = Dense(10, activation="softmax")(x)

model4 = Model(inputs, outputs)

model4.summary()

Model: "functional_4"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_5       │ (None, 32, 32, 3) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ random_flip_4       │ (None, 32, 32, 3) │          0 │ input_layer_5[0]… │
│ (RandomFlip)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ random_rotation_4   │ (None, 32, 32, 3) │          0 │ random_flip_4[0]… │
│ (RandomRotation)    │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ random_zoom_4       │ (None, 32, 32, 3) │          0 │ random_rotation_… │
│ (RandomZoom)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_66 (Conv2D)  │ (None, 32, 32,    │        864 │ random_zoom_4[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 32, 32,    │        128 │ conv2d_66[0][0]   │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation          │ (None, 32, 32,    │          0 │ batch_normalizat… │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ depthwise_conv2d_7  │ (None, 32, 32,    │        288 │ activation[0][0]  │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 32, 32,    │        128 │ depthwise_conv2d… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_1        │ (None, 32, 32,    │          0 │ batch_normalizat… │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 32)        │          0 │ activation_1[0][… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape (Reshape)   │ (None, 1, 1, 32)  │          0 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_4 (Dense)     │ (None, 1, 1, 8)   │        264 │ reshape[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_5 (Dense)     │ (None, 1, 1, 32)  │        288 │ dense_4[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multiply (Multiply) │ (None, 32, 32,    │          0 │ activation_1[0][… │
│                     │ 32)               │            │ dense_5[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_67 (Conv2D)  │ (None, 32, 32,    │        512 │ multiply[0][0]    │
│                     │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 32, 32,    │         64 │ conv2d_67[0][0]   │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_68 (Conv2D)  │ (None, 32, 32,    │      1,024 │ batch_normalizat

 Total params: 259,350 (1013.09 KB)

 Trainable params: 254,262 (993.21 KB)

 Non-trainable params: 5,088 (19.88 KB)

In [35]:
lr_schedule = tf.keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=1e-3, decay_steps=30*782
)

model4.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=lr_schedule),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [36]:
model4.fit(train_ds,epochs=30)

Epoch 1/30
782/782 ━━━━━━━━━━━━━━━━━━━━ 84s 86ms/step - accuracy: 0.4592 - loss: 1.4807
Epoch 2/30
782/782 ━━━━━━━━━━━━━━━━━━━━ 67s 86ms/step - accuracy: 0.5994 - loss: 1.1251
Epoch 3/30
782/782 ━━━━━━━━━━━━━━━━━━━━ 67s 86ms/step - accuracy: 0.6591 - loss: 0.9620
Epoch 4/30
782/782 ━━━━━━━━━━━━━━━━━━━━ 67s 86ms/step - accuracy: 0.7050 - loss: 0.8468
Epoch 5/30
782/782 ━━━━━━━━━━━━━━━━━━━━ 67s 86ms/step - accuracy: 0.7320 - loss: 0.7697
Epoch 6/30
782/782 ━━━━━━━━━━━━━━━━━━━━ 67s 86ms/step - accuracy: 0.7557 - loss: 0.6999
Epoch 7/30
782/782 ━━━━━━━━━━━━━━━━━━━━ 67s 86ms/step - accuracy: 0.7764 - loss: 0.6502
Epoch 8/30
782/782 ━━━━━━━━━━━━━━━━━━━━ 67s 86ms/step - accuracy: 0.7903 - loss: 0.6049
Epoch 9/30
782/782 ━━━━━━━━━━━━━━━━━━━━ 67s 86ms/step - accuracy: 0.8023 - loss: 0.5694
Epoch 10/30
782/782 ━━━━━━━━━━━━━━━━━━━━ 67s 86ms/step - accuracy: 0.8142 - loss: 0.5317
Epoch 11/30
782/782 ━━━━━━━━━━━━━━━━━━━━ 67s 85ms/step - accuracy: 0.8257 - loss: 0.5032
Epoch 12/30
782/782 ━━━━━━━━━━

In [37]:
model4.evaluate(test_ds)

157/157 ━━━━━━━━━━━━━━━━━━━━ 7s 36ms/step - accuracy: 0.8833 - loss: 0.3563


[0.35626983642578125, 0.8833000063896179]

## Model Comparison

| Model | Parameters | Speed (ms/step) | Test Accuracy | Test Loss |
|---|---|---|---|---|
| Inception | 753,826 | ~52 ms | 86.11% | 0.427 |
| ResNet | 2,799,850 | ~34 ms | 88.31% | 0.509 |
| MobileNet | 115,690 | ~25 ms | 70.71% | 0.856 |
| EfficientNet | 259,350 | ~87 ms | **88.33%** | **0.356** |

**Summary:** EfficientNet → best accuracy, slowest. MobileNet → fastest, lightest, lowest accuracy. Inception → good balance. ResNet → most parameters but not the best accuracy.